In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))


In [2]:
import pandas as pd
import numpy as np
import joblib
from src.data.ingest import build_universe, download_prices, clean_prices
from src.features.engineer import build_features
from src.models.optimise import optimise_portfolio


In [3]:
cache = Path("../data/raw/prices.parquet")
tickers = build_universe()
raw_df = download_prices(tickers, cache)
clean_df, _ = clean_prices(raw_df)

features_clean = pd.read_parquet("../data/processed/features.parquet")

FEATURE_COLS = [
    "momentum", "volatility_21d", "drawdown_52w",
    "relative_strength", "volume_ratio_20", "beta_252", "vix"
]

rf_return_v4     = joblib.load("../models/rf_return_v4.joblib")
rf_volatility_v4 = joblib.load("../models/rf_volatility_v4.joblib")

print("Ready.")


Loading cached data from ../data/raw/prices.parquet
Dropping 7 tickers: ['AAF.L', 'CCEP.L', 'EDV.L', 'HLN.L', 'MNG.L', 'MTLN.L', 'PSH.L']
Ready.


In [4]:
held_out = features_clean[
    features_clean.index.get_level_values("date") >= "2024-01-01"
]

snap = held_out[
    held_out.index.get_level_values("date") == pd.Timestamp("2024-01-31")
]

pred_returns = pd.Series(
    rf_return_v4.predict(snap[FEATURE_COLS]),
    index=snap.index.get_level_values("ticker")
)

pred_vols = pd.Series(
    rf_volatility_v4.predict(snap[FEATURE_COLS]),
    index=snap.index.get_level_values("ticker")
)

hist_returns = clean_df["Close"].pct_change(fill_method=None)

result = optimise_portfolio(
    predicted_returns=pred_returns,
    predicted_vols=pred_vols,
    historical_returns=hist_returns,
)

print(f"Converged: {result['converged']}")
print(f"Message:   {result['message']}")
print(f"Expected return:  {result['expected_return']:.4f}")
print(f"Expected vol:     {result['expected_vol']:.4f}")
print(f"\nTop 10 weights:")
print(result['weights'].sort_values(ascending=False).head(10))
print(f"\nWeight sum: {result['weights'].sum():.6f}")



Converged: True
Message:   Optimization terminated successfully
Expected return:  0.0037
Expected vol:     0.0645

Top 10 weights:
BP.L      0.100000
SGE.L     0.082656
SHEL.L    0.050034
JD.L      0.049884
SDR.L     0.048044
BAB.L     0.047333
BEZ.L     0.046771
ULVR.L    0.046695
CCH.L     0.044849
PSON.L    0.041134
dtype: float64

Weight sum: 1.000000


In [5]:
print(f"BP.L predicted return: {pred_returns['BP.L']:.4f}")
print(f"BP.L predicted vol:    {pred_vols['BP.L']:.4f}")
print(f"Mean predicted return: {pred_returns.mean():.4f}")
print(f"Mean predicted vol:    {pred_vols.mean():.4f}")


BP.L predicted return: 0.0022
BP.L predicted vol:    0.2406
Mean predicted return: 0.0034
Mean predicted vol:    0.2492


In [6]:
# Verify updated return structure
print(f"Converged:    {result['converged']}")
print(f"Weight sum:   {result['weights'].sum():.6f}")
print(f"Max weight:   {result['weights'].max():.4f}")
print(f"Expected return: {result['expected_return']:.4f}")
print(f"Expected vol:    {result['expected_vol']:.4f}")
print(f"\nKeys returned: {list(result.keys())}")
print(f"\ncorrelation_matrix shape: {result['correlation_matrix'].shape}")
print(f"correlation_matrix diagonal (should all be 1.0):")
print(np.diag(result['correlation_matrix'].values).round(6))


Converged:    True
Weight sum:   1.000000
Max weight:   0.1000
Expected return: 0.0037
Expected vol:    0.0645

Keys returned: ['weights', 'expected_return', 'expected_vol', 'mu', 'sigma_diag', 'cov_matrix', 'correlation_matrix', 'converged', 'message']

correlation_matrix shape: (93, 93)
correlation_matrix diagonal (should all be 1.0):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [7]:
corr_matrix = result['correlation_matrix']
print(f"BP.L vs SHEL.L correlation: {corr_matrix.loc['BP.L', 'SHEL.L']:.4f}")


BP.L vs SHEL.L correlation: 0.6488
